In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import plotly.express as px
import anndata as ad
# Fixed hardcoded path
import pathlib
PROJECT_ROOT = pathlib.Path(os.getcwd()).resolve().parents[1]
os.chdir(PROJECT_ROOT)
import plotly.io as pio
pio.renderers.default = "notebook"



In [20]:
df = pd.read_csv("FibroXplorer_Mouse_ss_Fibro_counts_corrected.csv", index_col=0)
metadata = pd.read_csv("FibroXplorer_Mouse_ss_Fibro_metadata_corrected.csv", index_col=0)
metadata['tissue'].unique()

array(['Bone', 'Visceral Adipose', 'Intestine', 'Heart',
       'Subcutaneous Adipose', 'Liver', 'Lymph Node', 'Lung', 'Muscle',
       'Pancreas', 'Skin', 'Spleen', 'Artery'], dtype=object)

In [21]:
df = df.T
metadata['tissue'] = metadata['tissue'].str.lower()
metadata['tissue'] = metadata['tissue'].replace({'visceral adipose':'fat','subcutaneous adipose':'fat'})

In [22]:
adata = ad.AnnData(X=df.values, obs=metadata)
adata.var_names = df.columns
mask = adata.obs['tissue'].isin({"fat", "skin", "heart", "lung", "pancreas", "intestine"})
adata = adata[mask].copy()

In [23]:
adata.obs['tissue'].value_counts()

tissue
fat          2000
intestine    1000
heart        1000
lung         1000
pancreas     1000
skin         1000
Name: count, dtype: int64

In [24]:
# sample 1000 cells from the 'fat' tissue

is_target = adata.obs['tissue'] == 'fat'
is_other = ~is_target
target_indices = np.where(is_target)[0]

np.random.seed(0)
sampled_target_indices = np.random.choice(target_indices, size=1000, replace=False)

final_indices = np.concatenate([sampled_target_indices, np.where(is_other)[0]])
adata = adata[final_indices, :].copy()

In [25]:
# normalize gene expression in each cell
adata.X = adata.X / adata.X.sum(axis=1)[:, None]
adata.X = np.log1p(adata.X)

In [26]:
# filter genes
mean_threshold = 8e-7
std_threshold = 5e-5
gene_means = adata.X.mean(axis=0)
gene_stds = adata.X.std(axis=0)

valid_cols = (gene_means > mean_threshold) & (gene_stds > std_threshold)
adata = adata[:, valid_cols].copy()


In [27]:
# z- score
gene_means = np.mean(adata.X, axis=0)
gene_stds = np.std(adata.X, axis=0)
# gene_stds[gene_stds == 0] = 1
adata.X = (adata.X - gene_means) / gene_stds

In [28]:
def export_to_df(adata, output_path):
    pd.DataFrame({"gene_name": adata.var_names}).to_csv(f"{output_path}/genes_names.csv", index=False)
    adata.obs.to_csv(f"{output_path}/metadata.csv", index=False)
    tissues = pd.Series(adata.obs['tissue'].unique())
    tissues.to_csv(f"{output_path}/tissues.csv", header=False, index=False)
    expr_df = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
    expr_df.to_csv(f"{output_path}/fibro.csv", index=False)

In [29]:
export_to_df(adata, "data/mouse_data/normalized_data")

In [30]:
def plot_PCA(df, metadata, coloring_param, output_path, arcs=None):
    pca = PCA(n_components=3)
    pca_result = pca.fit_transform(df)
    # Extract % variance explained for axis labels
    var_ratio = pca.explained_variance_ratio_ * 100
    labels = {
        'PC1': f'PC1 ({var_ratio[0]:.1f}%)',
        'PC2': f'PC2 ({var_ratio[1]:.1f}%)',
        'PC3': f'PC3 ({var_ratio[2]:.1f}%)',
        coloring_param: coloring_param
    }

    pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2', 'PC3'], index=df.index)
    pca_df = pca_df.join(metadata[coloring_param].astype(str))

    # Create 3D scatter plot
    fig = px.scatter_3d(
        pca_df,
        x='PC1', y='PC2', z='PC3',
        color=coloring_param,
        title=f'3D PCA colored by liver {coloring_param}',
        labels=labels,
        width=800,
        height=800
    )

    # Add archetypes as scatter3d
    def add_archetypes(fig, arc_df, color, name):
        fig.add_scatter3d(
            x=arc_df.iloc[:, 0],
            y=arc_df.iloc[:, 1],
            z=arc_df.iloc[:, 2],
            mode='markers+text',
            marker=dict(size=8, color=color, symbol='diamond'),
            textposition="top center",
            text=[f"{i+1}" for i in range(arc_df.shape[0])],
            name=name
        )


    if arcs is not None:
        add_archetypes(fig, arcs, 'gray', 'ParTI Archetypes')

    fig.update_traces(marker=dict(size=4), selector=dict(type='scatter3d'))
    fig.update_layout(
        font_color='black',
    scene=dict(
        xaxis=dict(backgroundcolor='white'),
        yaxis=dict(backgroundcolor='white'),
        zaxis=dict(backgroundcolor='white'),
        bgcolor='white'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

    fig.show()
    fig.write_html(f"{output_path}/PCA_mouse.html")


In [31]:
df1 = pd.read_csv("data/mouse_data/normalized_data/fibro.csv")
metadata1 = pd.read_csv("data/mouse_data/normalized_data/metadata.csv")
archetypes1 = pd.read_csv("arc.csv", header=None)


In [35]:
plot_PCA(df1, metadata1, 'tissue', "data/mouse_data", arcs=archetypes1)
